# Langchain- AI Agents Tools

In [1]:
%pip uninstall langchain langchain-core langchain-community -y

Found existing installation: langchain 1.0.1Note: you may need to restart the kernel to use updated packages.

Uninstalling langchain-1.0.1:
  Successfully uninstalled langchain-1.0.1
Found existing installation: langchain-core 1.0.1
Uninstalling langchain-core-1.0.1:
  Successfully uninstalled langchain-core-1.0.1
Found existing installation: langchain-community 0.3.29
Uninstalling langchain-community-0.3.29:
  Successfully uninstalled langchain-community-0.3.29


In [2]:
%pip install -q langchain==1.0.1 langchain-core==1.0.1 langchain-community==0.3.29 langchain-groq langchain-huggingface langchain-tavily python-dotenv sentence-transformers wikipedia

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")

if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
    print("GROQ_API_KEY loaded successfully")
else:
    print("GROQ_API_KEY is missing in your .env file")

GROQ_API_KEY loaded successfully


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [3]:
from langchain_groq import ChatGroq

# Use a Groq-supported model
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

print("LLM initialized successfully")

LLM initialized successfully


## Predefine Tools

In [4]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper()
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.run("LangChain"))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Milvus (vector database)
Summary: Milvus is a distributed vector database developed by Zilliz. It is available as both open-source software and a cloud service called Zilliz Cloud.
Milvus is an open-source project under the LF AI & Data Foundation and is distributed under the Apache License 2.0.



Page: Model Context Protocol
Summary: The Model Context Protocol (MCP) is an open standard and open-source framework introduced by Anthropic in November 2024 to standardize the way artificial intelligence (AI) systems like large language models (LLMs) integrate and share data with external tools, systems, and data sources. MCP provides a stan

In [6]:
from langchain_tavily import TavilySearch
import os

tavily_key = os.getenv("TAVILY_API_KEY")

if tavily_key:
    tool = TavilySearch(tavily_api_key=tavily_key)
    result = tool.invoke({"query": "What is the capital of France?"})
    print(result)
else:
    print("TAVILY_API_KEY not found in environment variables")

{'query': 'What is the capital of France?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.facebook.com/groups/302915185514582/posts/740821111723985', 'title': 'What is the capital of France? - Facebook', 'content': "PARIS it is the capital and largest city of France. Located on the Seine in the country's north, it is a major cultural and political centre of", 'score': 0.910998, 'raw_content': None}, {'url': 'https://home.adelphi.edu/~ca19535/page%204.html', 'title': 'Paris facts: the capital of France in history', 'content': 'Paris is the capital of France, the largest country of Europe with 550 000 km2 (65 millions inhabitants). Paris has 2.234 million inhabitants end 2011.', 'score': 0.8795718, 'raw_content': None}, {'url': 'https://www.instagram.com/reel/DWluHvojeX8?hl=en', 'title': 'Paris is the capital of France and one of the most visited ... - Instagram', 'content': 'Paris is the capital of France and one of the most visited cities i

## Create a Custom Tool

In [7]:
from langchain_core.tools import tool

@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

In [8]:
get_word_length.invoke("abc")

3

In [9]:
get_word_length.invoke("Sankalp Bankar")

14

In [10]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [11]:
@tool
def summisation(a: int, b: int) -> int:
    """Adding two numbers."""
    return a + b

In [12]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [13]:
print(summisation.name)
print(summisation.description)
print(summisation.args)

summisation
Adding two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [14]:
multiply.invoke({"a":10,"b":20})

200

In [15]:
summisation.invoke({"a":10,"b":20})

30

## Concept of Agents

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor
from langchain_core.utils.function_calling import convert_to_openai_function

# Define tools
tools = [multiply, summisation]

# Create function schema from tools
functions = [convert_to_openai_function(t) for t in tools]

# Bind tools to the model
model_with_tools = llm.bind(functions=functions)

# Proper prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided tools to answer questions."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create the agent chain using LCEL
from langchain_core.runnables import RunnablePassthrough
from langchain.agents.output_parsers import JSONAgentOutputParser

agent = prompt | model_with_tools | JSONAgentOutputParser()

print("Agent created successfully using LCEL")

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Users\LOQ\anaconda3\Lib\site-packages\langchain\agents\__init__.py)

In [20]:
import langchain
import langchain_core

print(langchain.__version__)
print(langchain_core.__version__)

1.0.1
1.0.1


In [19]:
from langchain.agents import AgentExecutor
from langchain_core.messages import HumanMessage

# Create executor with the agent and tools
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

# Invoke the agent
try:
    response = agent_executor.invoke({
        "input": "What is 10 multiplied by 20 and then add 5?"
    })
    print("\nAgent Response:")
    print(response)
except Exception as e:
    print(f"Error: {e}")
    print("\nAlternative approach - Direct tool usage:")
    # Fallback: use tools directly
    mult_result = multiply.invoke({"a": 10, "b": 20})
    print(f"10 * 20 = {mult_result}")
    add_result = summisation.invoke({"a": mult_result, "b": 5})
    print(f"{mult_result} + 5 = {add_result}")

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Users\LOQ\anaconda3\Lib\site-packages\langchain\agents\__init__.py)